In [2]:
# 라이브러리 & 기본 옵션

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from scipy import stats
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

# 1) 윈도우 기본 맑은고딕 파일 경로
font_path = r"C:\Windows\Fonts\malgun.ttf"

# 2) 폰트를 matplotlib에 "강제 등록"
fm.fontManager.addfont(font_path)

# 3) 등록한 폰트의 정확한 이름 가져오기
font_name = fm.FontProperties(fname=font_path).get_name()

# 4) matplotlib + seaborn 둘 다에 폰트 강제 적용 (seaborn이 덮는 문제 방지)
mpl.rcParams["font.family"] = font_name
mpl.rcParams["font.sans-serif"] = [font_name]
mpl.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid", rc={
    "font.family": font_name,
    "font.sans-serif": [font_name],
    "axes.unicode_minus": False
})

In [ ]:
# 0) 데이터 로드
import glob
import os

# 현재 폴더 안의 해당 CSV 파일들 모두 찾기
file_list = glob.glob("../data/소상공인시장진흥공단_상가(상권)정보_20251231/소상공인시장진흥공단_상가(상권)정보_*.csv")

print("파일 개수:", len(file_list))
display(file_list[:3])  # 일부 확인

파일 개수: 17


['../data/소상공인시장진흥공단_상가(상권)정보_20251231\\소상공인시장진흥공단_상가(상권)정보_강원_202512.csv',
 '../data/소상공인시장진흥공단_상가(상권)정보_20251231\\소상공인시장진흥공단_상가(상권)정보_경기_202512.csv',
 '../data/소상공인시장진흥공단_상가(상권)정보_20251231\\소상공인시장진흥공단_상가(상권)정보_경남_202512.csv']

In [4]:
# 1) 필요한 컬럼만 읽어서 합치기
df_list = []

for file in file_list:
    temp = pd.read_csv(
        file,
        usecols=["상권업종소분류명", "시도명", "시군구명"]
    )
    df_list.append(temp)

df_all = pd.concat(df_list, ignore_index=True)

In [5]:
# 2) 헬스장만 필터링
gym_df = df_all[df_all["상권업종소분류명"] == "헬스장"].copy()

# 시도별 헬스장 개수
gym_count_sido = (
    gym_df.groupby("시도명")
    .size()
    .reset_index(name="헬스장개수")
    .sort_values("헬스장개수", ascending=False)
)

# 시군구별 헬스장 개수
gym_count_sigungu = (
    gym_df.groupby(["시도명", "시군구명"])
    .size()
    .reset_index(name="헬스장개수")
    .sort_values(["시도명", "헬스장개수"], ascending=[True, False])
)

display(gym_count_sido)
display(gym_count_sigungu)

,시도명,헬스장개수
1,경기도,4492
8,서울특별시,4157
7,부산광역시,1049
11,인천광역시,911
2,경상남도,820
5,대구광역시,768
3,경상북도,617
15,충청남도,536
6,대전광역시,531
16,충청북도,491


,시도명,시군구명,헬스장개수
12,강원특별자치도,춘천시,136
8,강원특별자치도,원주시,112
0,강원특별자치도,강릉시,80
2,강원특별자치도,동해시,32
4,강원특별자치도,속초시,20
...,...,...,...
243,충청북도,증평군,9
239,충청북도,영동군,5
237,충청북도,단양군,2
236,충청북도,괴산군,1


In [ ]:
# 3) 헬스장 포함 그외 스포츠 시설까지 필터링
sports = ["헬스장", "골프 연습장", "기타 스포츠시설 운영업", "당구장", "볼링장", "수영장", "스쿼시/라켓볼장", "요가/필라테스 학원", "종합 스포츠시설", "탁구장", "태권도/무술학원", "테니스장"]

sports_df = df_all[df_all["상권업종소분류명"].isin(sports)].copy()

# 시도별 스포츠 시설 개수
sports_count_sido = (
    sports_df.groupby("시도명")
    .size()
    .reset_index(name="스포츠시설개수")
    .sort_values("스포츠시설개수", ascending=False)
)

# 시군구별 스포츠 시설 개수
sports_count_sigungu = (
    sports_df.groupby(["시도명", "시군구명"])
    .size()
    .reset_index(name="스포츠시설개수")
    .sort_values(["시도명", "스포츠시설개수"], ascending=[True, False])
)

display(sports_count_sido)
display(sports_count_sigungu)

,시도명,스포츠시설개수
1,경기도,23386
8,서울특별시,16303
11,인천광역시,4532
7,부산광역시,4424
2,경상남도,4392
3,경상북도,3442
5,대구광역시,3307
15,충청남도,3161
0,강원특별자치도,2834
16,충청북도,2720


,시도명,시군구명,스포츠시설개수
8,강원특별자치도,원주시,676
12,강원특별자치도,춘천시,553
0,강원특별자치도,강릉시,407
2,강원특별자치도,동해시,179
4,강원특별자치도,속초시,177
...,...,...,...
245,충청북도,증평군,54
241,충청북도,영동군,49
239,충청북도,단양군,42
240,충청북도,보은군,30


In [ ]:
# 4) 지역별(행정동) 성별 연령별 주민등록 인구수 데이터 로드
pop_df = pd.read_csv("../data/지역별(행정동) 성별 연령별 주민등록 인구수_20251231.csv", encoding="cp949")

# 데이터 확인
pop_df.head(1)

,행정기관코드,기준연월,시도명,시군구명,읍면동명,계,남자,여자,0세남자,1세남자,...,101세여자,102세여자,103세여자,104세여자,105세여자,106세여자,107세여자,108세여자,109세여자,110세이상 여자
0,1111051500,2025-12-31,서울특별시,종로구,청운효자동,10876,4909,5967,20,19,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 5) 필요한 컬럼만 추출
pop_use = pop_df[["시도명", "시군구명", "계"]].copy()

# 시도별 인구 합계
pop_sum_sido = (
    pop_use.groupby("시도명", as_index=False)["계"]
    .sum()
    .rename(columns={"계": "총 인구수"})
)

# 시군구별 인구 합계
pop_sum_sigungu = (
    pop_use.groupby(["시도명", "시군구명"], as_index=False)["계"]
    .sum()
    .rename(columns={"계": "총 인구수"})
)

display(pop_sum_sido.head())
display(pop_sum_sigungu.head())

,시도명,총 인구수
0,강원특별자치도,1508500
1,경기도,13730135
2,경상남도,3207383
3,경상북도,2506526
4,광주광역시,1392013


,시도명,시군구명,총 인구수
0,강원특별자치도,강릉시,206237
1,강원특별자치도,고성군,26845
2,강원특별자치도,동해시,86333
3,강원특별자치도,삼척시,60518
4,강원특별자치도,속초시,79636


In [ ]:
# 6) 헬스장 개수 + 시군구별 인구 합계 조인

# 시도별
gym_pop_sido = pd.merge(
    gym_count_sido,
    pop_sum_sido,
    on=["시도명"],
    how="left"
)
display(gym_pop_sido.head())
print(gym_pop_sido.isna().sum())

# 시군구별
gym_pop_sigungu = pd.merge(
    gym_count_sigungu,
    pop_sum_sigungu,
    on=["시도명", "시군구명"],
    how="left"
)
display(gym_pop_sigungu.head())
print(gym_pop_sigungu.isna().sum())

,시도명,헬스장개수,총 인구수
0,경기도,4492,13730135
1,서울특별시,4157,9299548
2,부산광역시,1049,3241600
3,인천광역시,911,3051961
4,경상남도,820,3207383


시도명      0
헬스장개수    0
총 인구수    0
dtype: int64


,시도명,시군구명,헬스장개수,총 인구수
0,강원특별자치도,춘천시,136,285234
1,강원특별자치도,원주시,112,363194
2,강원특별자치도,강릉시,80,206237
3,강원특별자치도,동해시,32,86333
4,강원특별자치도,속초시,20,79636


시도명      0
시군구명     0
헬스장개수    0
총 인구수    0
dtype: int64


In [ ]:
# 7) 스포츠 시설 개수 + 시군구별 인구 합계 조인

# 시도별
sports_pop_sido = pd.merge(
    sports_count_sido,
    pop_sum_sido,
    on=["시도명"],
    how="left"
)
display(sports_pop_sido.head())
print(sports_pop_sido.isna().sum())

# 시군구별
sports_pop_sigungu = pd.merge(
    sports_count_sigungu,
    pop_sum_sigungu,
    on=["시도명", "시군구명"],
    how="left"
)
display(sports_pop_sigungu.head())
print(sports_pop_sigungu.isna().sum())

,시도명,스포츠시설개수,총 인구수
0,경기도,23386,13730135
1,서울특별시,16303,9299548
2,인천광역시,4532,3051961
3,부산광역시,4424,3241600
4,경상남도,4392,3207383


시도명        0
스포츠시설개수    0
총 인구수      0
dtype: int64


,시도명,시군구명,스포츠시설개수,총 인구수
0,강원특별자치도,원주시,676,363194
1,강원특별자치도,춘천시,553,285234
2,강원특별자치도,강릉시,407,206237
3,강원특별자치도,동해시,179,86333
4,강원특별자치도,속초시,177,79636


시도명        0
시군구명       0
스포츠시설개수    0
총 인구수      0
dtype: int64


In [21]:
# 8) 저장
save_path = os.path.join("..", "data")

gym_pop_sido.to_csv(os.path.join(save_path, "시도별_헬스장개수&총인구수.csv"), index=False, encoding="utf-8-sig")
gym_pop_sigungu.to_csv(os.path.join(save_path, "시군구별_헬스장개수&총인구수.csv"), index=False, encoding="utf-8-sig")
sports_pop_sido.to_csv(os.path.join(save_path, "시도별_스포츠시설개수&총인구수.csv"), index=False, encoding="utf-8-sig")
sports_pop_sigungu.to_csv(os.path.join(save_path, "시군구별_스포츠시설개수&총인구수.csv"), index=False, encoding="utf-8-sig")